In [ ]:
! pip install torch==2.9.0 transformers==4.57.3 pandas==2.2.2 numpy==2.0.2

### Greedy vs Beam

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import pandas as pd

# Model setup
model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=dtype, trust_remote_code=True
).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!\n")

# Helpers
def get_top_n_predictions(logits, n=5):
    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, n)
    return [(tokenizer.decode([int(i)]), float(p)) for i, p in zip(top_ids, top_probs)]

def visualize_step(step, text, preds, chosen):
    print("\n" + "=" * 80)
    print(f"STEP {step}")
    print("=" * 80)
    print(f"Current text: \"{text}\"")
    print("\nTop-n next-token predictions:")

    df = pd.DataFrame({
        "Rank": range(1, 6),
        "Token": [repr(t) for t, _ in preds],
        "Probability": [f"{p:.4f}" for _, p in preds],
    })
    print(df.to_string(index=False))
    print(f"\nSelected token: {repr(chosen)}")

# Step-by-step generation
def generate_step_by_step(
    prompt,
    steps=5
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    current_text = prompt

    print("=" * 80)
    print("AUTOREGRESSIVE GENERATION (GREEDY)")
    print("=" * 80)
    print(f"Prompt: \"{prompt}\"")

    for step in range(steps):
        with torch.no_grad():
            logits = model(input_ids).logits[0, -1]

        next_id = torch.argmax(logits).view(1, 1)
        display_logits = logits

        preds = get_top_n_predictions(display_logits)
        token_str = tokenizer.decode(next_id[0])

        visualize_step(step + 1, current_text, preds, token_str)

        input_ids = torch.cat([input_ids, next_id], dim=-1)
        current_text += token_str

    print("\nFINAL OUTPUT:")
    print(current_text)
    print("=" * 80 + "\n")

def generate_beam_step_by_step(
    prompt,
    steps=5,
    num_beams=3,
    length_penalty=1.0
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # Each beam: (input_ids, cumulative_logprob)
    beams = [(input_ids, 0.0)]

    print("=" * 80)
    print("AUTOREGRESSIVE GENERATION (BEAM SEARCH)")
    print("=" * 80)
    print(f"Prompt: \"{prompt}\"")
    print(f"num_beams: {num_beams}, length_penalty: {length_penalty}\n")

    for step in range(steps):
        all_candidates = []

        print(f"\n{'='*30} STEP {step+1} {'='*30}")

        for beam_idx, (beam_ids, beam_score) in enumerate(beams):
            with torch.no_grad():
                logits = model(beam_ids).logits[0, -1]

            log_probs = F.log_softmax(logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, num_beams)

            print(f"\nBeam {beam_idx+1} prefix:")
            print(tokenizer.decode(beam_ids[0], skip_special_tokens=True))

            for log_p, token_id in zip(top_log_probs, top_ids):
                new_ids = torch.cat(
                    [beam_ids, token_id.view(1, 1)], dim=-1
                )
                new_score = beam_score + log_p.item()

                all_candidates.append((new_ids, new_score))

                print(
                    f"  + {repr(tokenizer.decode([int(token_id)]))} "
                    f"(logP={log_p.item():.4f})"
                )

        # Rank all candidates
        all_candidates.sort(
            key=lambda x: x[1] / (x[0].shape[-1] ** length_penalty),
            reverse=True
        )

        # Keep top beams
        beams = all_candidates[:num_beams]

        print("\nSelected beams:")
        for i, (ids, score) in enumerate(beams):
            print(
                f"[{i+1}] score={score:.4f} | "
                f"{tokenizer.decode(ids[0], skip_special_tokens=True)}"
            )

    print("\nFINAL OUTPUT (BEST BEAM):")
    best_ids, best_score = beams[0]
    print(tokenizer.decode(best_ids[0], skip_special_tokens=True))
    print("=" * 80 + "\n")


# DEMOS

# Greedy
generate_step_by_step(
    "The cat sat on the",
    steps=5,
)

# Beam
generate_beam_step_by_step(
    "The cat sat on the",
    steps=5,
    num_beams=3
)

Model loaded!

AUTOREGRESSIVE GENERATION (GREEDY)
Prompt: "The cat sat on the"

STEP 1
Current text: "The cat sat on the"

Top-n next-token predictions:
 Rank     Token Probability
    1    ' mat'      0.8426
    2    ' cat'      0.0247
    3  ' fence'      0.0110
    4      '\n'      0.0094
    5 ' window'      0.0046

Selected token: ' mat'

STEP 2
Current text: "The cat sat on the mat"

Top-n next-token predictions:
 Rank   Token Probability
    1     '.'      0.2193
    2   '.\n'      0.1617
    3     ','      0.1018
    4  ' and'      0.0855
    5 '.\n\n'      0.0657

Selected token: '.'

STEP 3
Current text: "The cat sat on the mat."

Top-n next-token predictions:
 Rank   Token Probability
    1  ' The'      0.1806
    2     ' '      0.1752
    3 ' What'      0.0722
    4    ' ('      0.0445
    5   ' It'      0.0346

Selected token: ' The'

STEP 4
Current text: "The cat sat on the mat. The"

Top-n next-token predictions:
 Rank   Token Probability
    1  ' cat'      0.5728
    2 

### Generation Parameters

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=dtype,
    trust_remote_code=True
).to(device)

model.eval()

# Safety: set pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!")
print("Loaded tokenizer:", tokenizer.name_or_path)
print("Loaded model:", model.config._name_or_path)
print("Device:", device, "| dtype:", dtype, "\n")

# Sample prompts to try
SAMPLE_PROMPTS = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The recipe for chocolate chip cookies requires"
]

def generate_text(prompt, temperature=1.0, top_p=1.0, top_k=50, max_length=50, num_return=1):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_new_tokens=max_length,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            num_return_sequences=num_return,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

def print_comparison(prompt, results_dict):
    """Pretty print comparison of different sampling strategies"""
    print("="*80)
    print(f"PROMPT: '{prompt}'")
    print("="*80)

    for config, texts in results_dict.items():
        print(f"\n{config}:")
        print("-" * 80)
        for i, text in enumerate(texts, 1):
            # Remove prompt from output
            generated = text[len(prompt):].strip()
            print(f"  Output {i}: {generated}")
    print("\n")

# EXPERIMENT 1: Temperature Comparison
print("\n" + "EXPERIMENT 1: TEMPERATURE COMPARISON" + "\n")

prompt = SAMPLE_PROMPTS[0]

temp_results = {
    "Temperature = 0.01 (Very low randomness)": generate_text(prompt, temperature=0.01),
    "Temperature = 0.7 (Balanced)": generate_text(prompt, temperature=0.7),
    "Temperature = 1.5 (High Creativity)": generate_text(prompt, temperature=1.5),
}

print_comparison(prompt, temp_results)

print("OBSERVATION:")
print("  - Low temp: Repetitive, predictable outputs")
print("  - Medium temp: Varied but coherent outputs")
print("  - High temp: Very diverse, sometimes nonsensical outputs")

# EXPERIMENT 2: Top-p (Nucleus Sampling) Comparison
print("\n" + "EXPERIMENT 2: TOP-P (NUCLEUS SAMPLING) COMPARISON" + "\n")

prompt = SAMPLE_PROMPTS[1]

topp_results = {
    "Top-p = 0.5 (Conservative)": generate_text(prompt, temperature=0.8, top_p=0.5),
    "Top-p = 0.9 (Balanced)": generate_text(prompt, temperature=0.8, top_p=0.9),
    "Top-p = 1.0 (Full Distribution)": generate_text(prompt, temperature=0.8, top_p=1.0),
}

print_comparison(prompt, topp_results)

print("OBSERVATION:")
print("  - Low top-p: Focuses on high-probability tokens (safer)")
print("  - High top-p: Includes lower-probability tokens (more diverse)")
print("  - top_p works by adapting the cutoff based on probability distribution")

# EXPERIMENT 3: Top-k Comparison
print("\n" + "EXPERIMENT 3: TOP-K COMPARISON" + "\n")

prompt = SAMPLE_PROMPTS[2]

topk_results = {
    "Top-k = 10 (Very Restrictive)": generate_text(prompt, temperature=0.8, top_k=10),
    "Top-k = 50 (Balanced)": generate_text(prompt, temperature=0.8, top_k=50),
    "Top-k = 100 (Permissive)": generate_text(prompt, temperature=0.8, top_k=100),
}

print_comparison(prompt, topk_results)

print("OBSERVATION:")
print("  - Low top-k: More focused, less diversity")
print("  - High top-k: More tokens to choose from, more variety")
print("  - Unlike top-p, top-k always considers exactly k tokens")

Model loaded!
Loaded tokenizer: Qwen/Qwen2.5-0.5B
Loaded model: Qwen/Qwen2.5-0.5B
Device: cpu | dtype: torch.float32 


EXPERIMENT 1: TEMPERATURE COMPARISON

PROMPT: 'The future of artificial intelligence is'

Temperature = 0.01 (Very low randomness):
--------------------------------------------------------------------------------
  Output 1: in the hands of the people. The future of artificial intelligence is in the hands of the people. The future of artificial intelligence is in the hands of the people. The future of artificial intelligence is in the hands of the people. The future of artificial

Temperature = 0.7 (Balanced):
--------------------------------------------------------------------------------
  Output 1: here. And it is one that will change the way we work and interact with the world. AI is changing every industry, and you can be one of them. Here are five ways you can take advantage of AI and stay ahead of the competition.

Temperature = 1.5 (High Creativity):
---------